# TUTORIAL: The annular combustor, from model to a real-data digital twin

## 1. The azimuthal thermoacoustic low-order model

The first azimuthal mode pair of an annular combustion chamber is described by
the acoustic pressure

$$
p(\theta, t) = \eta_a(t)\cos\theta + \eta_b(t)\sin\theta,
$$

whose mode amplitudes obey two coupled oscillators with a growth rate $\nu$, a
resistive asymmetry $c_2\beta$, a saturation $\kappa$ and a reactive asymmetry
$(\epsilon, \Theta_\epsilon)$. The balance between $\nu$ and $c_2\beta$ sets the
regime:

- spinning modes: $\nu = 30$, $c_2\beta = 5$
- standing modes: $\nu = 0$, $c_2\beta = 50$
- mixed modes: $\nu = 20$, $c_2\beta = 18$

## The model now lives in `dynamodels`

The model itself, and the tutorial that walks through its mode pair, its phase
space and its azimuthal pressure field, moved to
[`dynamodels`](https://andreanovoa.github.io/dynamodels/), the modelling core
that `romda` builds on:

- **Live tutorial:**
  [The annular combustor](https://andreanovoa.github.io/dynamodels/tutorials/tutorial_annular.html)
  — the mode pair $(\eta_a, \eta_b)$ and its phase space, the pressure at four
  microphones around the annulus, and an animation of the azimuthal pressure
  field.
- **Model reference:**
  [Annular combustor](https://andreanovoa.github.io/dynamodels/models/annular/)
  — the three named regimes (`case=`) and the full API.

Sections 2 and 3 below explore the experimental data and assimilate it into this
model, which is what `romda` adds on top.

`romda.models.physical` re-exports the `dynamodels` models, so existing code
keeps working unchanged:

In [ ]:
from romda.models.physical import Annular

case = Annular(nu=20., c2beta=18.)  # the mixed mode

state, t_ = case.time_integrate(int(case.t_transient * 3 / case.dt))
case.update_history(state, t_)
case.visualize_observable_hist()

## 2. Real thermoacoustic data from an annular combustor

The experiments are those of
[Indlekofer et al. (2022)](https://doi.org/10.1017/jfm.2022.468) on a
hydrogen-based annular combustor.

In [ ]:
from romda.models.physical import Annular
from romda.utils import *

data_folder, results_folder, figs_folder = set_working_directories('annular/')

get_annular_data(data_folder) # Download the data if not already present




The data available in data_folder are acoustic pressure measurements at the azimuthal locations $ \theta = 0^\circ, 60^\circ, 120^\circ, 240^\circ.$
 
Each .mat data file represents an experiment at different equivalence ratios $\Phi = 0.4875, ..., 0.5750$ (by steps of $0.0125$); and the file includes the variables
- y_raw: raw acoustic pressure recordings 
- y_filtered: post-processed acoustic pressure (mean offset correction and bandpass filter)
- t: timestamp of the recordings


In [ ]:

# Define .mat file names and equivalence ratios
ERs = 0.4875 + np.arange(0, 4) * 0.025
files = [data_folder + 'ER_{}.mat'.format(ER) for ER in ERs]



Using the functions "nu_from_ER" and "c2beta_from_ER" we can compute fore each equivalence ratio the linear growth rate $\nu$ and the asymmetry parameter group $c_2\beta$, which allows us to replicate Fig. 9 in [Indlekofer et al. (2022)](https://doi.org/10.1017/jfm.2022.468).



In [ ]:
NUs, C2Bs = [], []
for ER in ERs:
    NUs.append(Annular.nu_from_ER(ER))
    C2Bs.append(Annular.c2beta_from_ER(ER))

fig = plt.figure(figsize=(4, 3))
plt.axhline(y=0, color='lightgrey', ls='--')
plt.plot(ERs, NUs, '-', color='C0', label='$\\nu$')
plt.plot(ERs, C2Bs, '-', color='r', label=Annular.alpha_labels['c2beta'])
plt.xlabel(r'Equivalence ratio, $\Phi$')
plt.ylabel('[s$^{-1}$]')
plt.ylim([-50, 50])
plt.legend(loc='lower right');



Let's visualize the power spectral density (PSD) of the acoustic pressure for each equivalence ratio.

In [ ]:
import scipy.io as sio

def fun_PSD(X):
    if X.shape[0] > X.shape[1]:
        X = X.T
    PSD = []
    for x in X:
        yt = np.fft.fft(x)
        PSD.append(2.0 / X.shape[-1] * np.abs(yt[0:X.shape[-1] // 2]))
    return PSD

N, upsample = 1000, 2

fig1 = plt.figure(figsize=(10, 4))
axs = fig1.subplots(2, len(ERs)//2, sharey='all', sharex='all')
for ER, name, ax in zip(ERs, files, axs.ravel()):
    mat = sio.loadmat(name)
    y_raw, t_obs = [mat[key] for key in ['y_raw', 't']]
    y_raw = y_raw[-N::upsample]
    if ER == ERs[0]:
        Nq = y_raw.shape[-1]
        t_obs = t_obs.squeeze()[::upsample]
        dt = t_obs[1]-t_obs[0]
        f = np.linspace(0.0, 1.0 / (2.0 * dt), (N//upsample) // 2)
        colors = plt.cm.afmhot(np.linspace(0,0.7,Nq))[::-1]
        
    psd = fun_PSD(X=y_raw)
    for psd_, c, lw in zip(psd, colors, [3, 2, 1.2, 0.8]):
        ax.plot(f, psd_, color=c, lw=lw)
    ax.set(title='$\\Phi={}$'.format(ER))
    
leg = ['$p(\\theta={}^\\circ)$'.format(deg) for deg in [0, 60, 120, 240]]
axs[0, -1].legend(leg, loc='upper left', bbox_to_anchor=(1, 1))
[ax.set(xlabel='$f$') for ax in axs[1, :]]
[ax.set(ylabel='PSD', xlim=[100, 3000], ylim=[-10, 600]) for ax in axs[:, 0]];


We focus now on a single experiment. We can visualize the instantaneous pressure at each microphone. 
The figure shows how 
1. the post-processed data (i.e., y_filtered) is zero-mean, 
2. the post-processed signals are smoother, and
3. the raw pdfs are wider and more flat.

In [ ]:
# Select an equivalence ratio
idx = -1
ER, name = ERs[idx], files[idx]

# Load data
mat = sio.loadmat(name)
y_raw, y_filter, t = [mat[key] for key in ['y_raw', 'y_filtered', 't']]
t = t.squeeze()
Nq = y_raw.shape[-1]


In [ ]:

from scipy.signal import find_peaks

upsample = 2
N_max = int(20 / (t[1] - t[0])) 

y_raw_, y_filter_, t_ = [xx[0:N_max:upsample] for xx in [y_raw, y_filter, t]]


fig1 = plt.figure(figsize=(13, 1.5 * Nq), layout="tight")
titles = ['Raw', 'Post-processed']
labels_y = ['$p(\\theta={}^\\circ)$'.format(th) for th in [0, 60, 120, 240]]
cols = ['tab:blue', 'mediumseagreen']

max_y, N_zoom = np.max(y_raw_), int(0.0201 // (t_[1] - t_[0]))
# Plot zoomed timeseries of raw, post-processed and noise
axs = fig1.subplots(Nq, 5, sharex='col', sharey='all', width_ratios=[1.5, 1, 1.5, 1, 0.5])
ax_raw , ax_filter, ax_pdf= axs[:, :2], axs[:, 2:4], axs[:, -1]

for ax, yy, ttl, c in zip([ax_raw, ax_filter], [y_raw_, y_filter_], titles, cols):
    ax[0, 0].set(title=ttl)
    ax[-1, 0].set(xlabel='$t$', xlim=[t_[0], t_[-1]])
    ax[-1, -1].set(xlabel='$t$', xlim=[t_[-N_zoom], t_[-1]])
    for qi in range(Nq):
        ax[qi, 0].plot(t_[:-N_zoom], yy[:-N_zoom, qi], color=c)
        ax[qi, 1].plot(t_[-N_zoom:], yy[-N_zoom:, qi], color=c)
        ax[qi, 1].axhline(np.mean(yy[:, qi]), color=c) 
        if ttl == titles[0]:
            ax[qi, 0].set(ylabel=labels_y[qi])

bins = np.arange(-max_y, max_y + 0.01 * max_y, 0.01 * max_y)
for yy, ttl, c, a in zip([y_raw_, y_filter_], titles, cols, [1, 0.8]):
    ax_pdf[0].set(title='Peaks count')
    ax_pdf[-1].set(xlabel='pdf')
    for qi in range(Nq):
        peaks = find_peaks(abs(yy[:, qi]))[0]
        ax_pdf[qi].hist(yy[peaks, qi], bins=bins, density=True, orientation='horizontal', color=c, alpha=a)


## 3. A real-time digital twin of the annular combustor

We can now put everything we have learned together. Data assimilation using real experimental data

We develop a digital twin of the hydrogen-based annular combustor. We want to estimate the LOM parameters and states from raw experimental data from microphones.  Because the raw data may be biased, we need to model the bias in both, measurement data and model.


### 3.1. Load the data 
Create the reference truth and the observations.



In [ ]:
from romda.models.physical import Annular
from romda.observations import Observations
import os

ER = 0.4875 + 0.025 # 0.4875 + np.arange(0, 4) * 0.025

t_start = Annular.t_transient
t_stop = t_start + Annular.t_CR * 15


truth = Observations(model = os.path.join(data_folder, 'ER_{}'.format(ER)),
                     t_start = t_start,
                     t_stop = t_stop,
                     Nt_obs = 35,
                     t_max = t_stop + Annular.t_transient,
                     add_noise = False
                     )


In [ ]:
Observations.plot_truth(truth, Nq=4, fig_width=12, window=0.025, f_max=10000)

### 3.2. Define the forecast model
This is the physical model which we will use to model the true data.
Here, we select the filter parameters and create ensemble

*The function ```create_ensemble``` consists of* 

```
alpha0_mean = dict()
for alpha, lims in alpha0.items():
    alpha0_mean[alpha] = 0.5 * (lims[0] + lims[1])

ensemble = Annular(**alpha0_mean)

filter_params = dict(m= 20, 
                     std_psi=0.3,
                     std_a=alpha0)

# Forecast model to initialise the ensemble after transient
state, t_ = ensemble.time_integrate(int(ensemble.t_CR / ensemble.dt))
ensemble.update_history(state[-1], reset=True)

ensemble.init_ensemble(**filter_params)
ensemble.close()
```

In [ ]:
from romda.estimators import rBA_EnKF, EnKF, EnSRKF
import numpy as np

alpha0 = dict(nu=(-15., 30.),
              c2beta=(10, 50),
              kappa=(1.E-4, 2.E-4),
              epsilon=(5e-3, 8e-3),
              omega=(1090 * 2 * np.pi, 1095 * 2 * np.pi),
              theta_b=(0.5, 0.7),
              theta_e=(0.4, 0.6)
              )

ensemble = rBA_EnKF(parent_model=Annular(dt=truth.dt),
                    m=20,
                    std_phi=0.3,
                    std_alpha=alpha0,
                    distribution_alpha='uniform',
                    )

In [ ]:
# Visualize ensemble initialization
ensemble.visualize_state(reference_a={'kappa': 1e-4, 'omega': 2 * np.pi, 'epsilon': 1e-3})

### 3.3. Train an ESN to model the model bias
The procedure is the following

&emsp; i. Initialise ESN Bias class object
&emsp; ii. Create synthetic bias to use as training data 
&emsp; iii. Train the ESN
&emsp; iv. Create washout data

<br>

**4.1. Initialise the ESN**

In [ ]:
from romda.bias_estimators import ESN_bias
import numpy as np

training_data_filename = f'{results_folder}/ESN_train_data_annular_raw'

bias_estimator = ESN_bias(rom=ensemble.model,
                          reference_data=truth,
                          training_data_filename=training_data_filename,
                          upsample=5,
                          N_units=50,
                          N_wash=10,
                          t_train=ensemble.model.t_transient / 3.,
                          t_test=ensemble.model.t_CR * 2,
                          t_val=ensemble.model.t_CR * 2,
                          # Training data generation options
                          augment_data=True,
                          biased_observations=True,
                          correlation_based_training=True,
                          N_folds=4,
                          L=20,
                          std_alpha=alpha0,
                          # Hyperparameter search ranges
                          rho_range=(0.5, 1.),
                          sigma_in_range=(np.log10(1e-5), np.log10(1e1)),
                          tikh_range=[1e-12, 1e-9],
                          N_ens=ensemble.m,
                          )


**4.2 Create training data**

The details of the code inside ```create_bias_training_dataset()``` function is explained in the tutorial ```Class_Bias.ipynb```.

**4.3. Train the ESN**

The training convergence, hyperparameter optimization and testing results are saved in a pdf file in *figs_ESN* folder.

**4.4. Create washout data**

We retrieve from the raw data a ```N_wash``` number of observations to use for initialising the ESN, i.e., to perform the washout. 
The ESN initialization must be before the fist observation.

```
from create import create_washout
wash_t, wash_obs = create_washout(ensemble.bias, t=t_true, y_raw=y_raw)
```

In [ ]:
ensemble_BA = ensemble.copy()
ensemble_BA.bias = bias_estimator.copy()

### 3.4. Apply data assimilation
We now have all the ingredients to start our data assimilation algorithm.

In [ ]:
# Observation error covariance matrix
std_obs = 0.05
Cdd = np.diag(std_obs * np.ones(ensemble.model.Nq)) * np.max(abs(truth.y_obs), axis=0) ** 2

t_extra = ensemble.model.t_CR * 10

out = []
ks = [0, 5.]  # bias regularization factors

for kk in ks:
    ens = ensemble_BA.copy()
    ens.regularization_factor = kk
    ens.filter.gamma = kk

    for d, t_d in zip(truth.y_obs, truth.t_obs):
        ens.forecast_step(t_end=t_d)
        ens.analysis_step(d=d, Cdd=Cdd.copy())

    ens.forecast_step(t_end=truth.t_obs[-1] + t_extra, close=True)
    out.append(ens)

In [ ]:
for ens in out:
    ens.visualize_history(truth=truth, plot_members=False, dims=[0, 1])
    ens.bias.visualize_bias_and_innovations(plot_members=True)